# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load, inspect, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is distributed following the [Croissant](https://mlcommons.org/croissant) metadata standard and is accessed entirely by its schema URL.

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get the metadata as a dict
metadata = dataset.metadata.to_json()
print("Dataset Name:", metadata['name'])
print("Description:", metadata['description'])

## 2. Data Overview
List the available record sets, their `@id`s, and the fields inside each record set. Croissant datasets reference all entity IDs via their `@id`. This ensures precise selection and extraction steps.

In [ ]:
# Explore the main record sets and their field IDs
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    print(f"  Name: {rs.get('name','')}")
    print(f"  Description: {rs.get('description','')}")
    # List all fields (columns) and their IDs
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields:")
    for f in fields:
        fid = f['@id'] if isinstance(f, dict) else f
        print(f"    - {fid}")
    print()
# List all record set @ids for easy copy-paste
record_set_ids = [rs['@id'] for rs in record_sets]
print("record_set_ids (for later use):", record_set_ids)

## 3. Data Extraction
Load examples from each available record set using its `@id`, then load all records from the primary protein data table.

**Note:** All field/column and record set references will use their Croissant `@id` (e.g. `cr:ProteinTable`, `cr:accession`, etc.).

In [ ]:
# Select the main protein data record set by searching for an identifier containing "Protein" or based on README/metadata
main_record_set_id = None
for rs in dataset.record_sets:
    if 'protein' in rs['@id'].lower() or rs['name'].lower().startswith('protein'):
        main_record_set_id = rs['@id']
        break
if main_record_set_id is None:
    # Default: Pick first if only one
    main_record_set_id = record_set_ids[0]
print(f"Using main record set: {main_record_set_id}")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  {len(df)} records loaded, columns: {list(df.columns)}\n")

# Show example records and column names for the main record set
print(f"Columns for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let us select a relevant numeric field (e.g., normalized abundance, coverage, peptide count, etc.), filter and normalize it, and group by a categorical attribute such as protein accession description. All field names are looked up using their `@id`.

In [ ]:
# Identify a numeric field for filtering and normalization
numeric_fields = ['cr:coverage', 'cr:unique_peptides', 'cr:peptide_count', 'cr:mw', 'cr:pi', 'cr:norm_abundance_sample1', 'cr:norm_abundance_sample2']

# Find which numeric field exists in this dataset
main_df = dataframes[main_record_set_id]

chosen_numeric_field = None
for f in numeric_fields:
    if f in main_df.columns:
        chosen_numeric_field = f
        break
if chosen_numeric_field is None:
    raise ValueError("No known numeric field ID found in dataset.")

print(f"Using numeric field {chosen_numeric_field}")

# Set a sample threshold for exploration (quantile-based for demo)
if main_df[chosen_numeric_field].dtype == object:
    # Convert if necessary
    main_df[chosen_numeric_field] = pd.to_numeric(main_df[chosen_numeric_field], errors='coerce')

threshold = main_df[chosen_numeric_field].quantile(0.9)
filtered_df = main_df[main_df[chosen_numeric_field] > threshold]
print(f"Filtered records with {chosen_numeric_field} > {threshold:.2f} (top 10%): {len(filtered_df)} rows")

# Normalize the filtered values
filtered_df = filtered_df.copy()
filtered_df[f"{chosen_numeric_field}_normalized"] = (
    (filtered_df[chosen_numeric_field] - filtered_df[chosen_numeric_field].mean()) /
    filtered_df[chosen_numeric_field].std()
)
print(filtered_df[[chosen_numeric_field, f"{chosen_numeric_field}_normalized"]].head())

# Try grouping by a categorical field
possible_group_fields = ['cr:accession', 'cr:description', 'cr:modification']
group_field = None
for gf in possible_group_fields:
    if gf in main_df.columns:
        group_field = gf
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[chosen_numeric_field].mean().reset_index()
    print(f"Grouped mean {chosen_numeric_field} by {group_field} (top 5):")
    print(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization
Show a histogram for the selected numeric field, and a bar plot of group means for the main protein/categorical field. This step uses field `@id` names for all columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of selected normalized numeric field
plt.figure(figsize=(8,5))
sns.histplot(main_df[chosen_numeric_field].dropna(), bins=30)
plt.title(f'Histogram of {chosen_numeric_field}')
plt.xlabel(chosen_numeric_field)
plt.ylabel('Frequency')
plt.show()

# If grouped data available, show top N bar plot
if group_field and not filtered_df.empty:
    grouped_sorted = grouped_df.sort_values(by=chosen_numeric_field, ascending=False).head(20)
    plt.figure(figsize=(10,5))
    sns.barplot(y=group_field, x=chosen_numeric_field, data=grouped_sorted)
    plt.title(f'Top 20 {group_field} by mean {chosen_numeric_field}')
    plt.xlabel(chosen_numeric_field)
    plt.ylabel(group_field)
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and inspected a Croissant metadata-driven dataset (FAIR^2)
- Explored available record sets, fields, and their Croissant `@id`s
- Extracted protein data and showcased normalization and filtering using field identifiers
- Produced visualizations using the Croissant logical field names

**This workflow ensures reproducible, FAIR referencing of scientific and bioinformatics data structures.**

For further exploration, consult the Croissant metadata structure for any additional relations (e.g., post-translational modifications or additional sample comparisons) inside this dataset.